# Data Cleaning & Feature Engineering

## Project

E-Commerce Sales Intelligence Platform

### Objective

This notebook cleans, validates, and prepares the Olist Brazilian E-Commerce dataset for SQL modeling, business analysis, and Power BI dashboards.

Prepared By: Tsering Gurung

In [2]:
import pandas as pd
import numpy as np
from pathlib import Path

In [3]:
data_path = Path("../data/raw")

datasets = {}

for file in sorted(data_path.glob("*.csv")):
    datasets[file.stem] = pd.read_csv(file)

print(f"Loaded {len(datasets)} datasets.")

Loaded 9 datasets.


In [4]:
customers = datasets["olist_customers_dataset"]
orders = datasets["olist_orders_dataset"]
items = datasets["olist_order_items_dataset"]
payments = datasets["olist_order_payments_dataset"]
products = datasets["olist_products_dataset"]
reviews = datasets["olist_order_reviews_dataset"]
sellers = datasets["olist_sellers_dataset"]
geolocation = datasets["olist_geolocation_dataset"]
translations = datasets["product_category_name_translation"]

In [5]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype
---  ------                         --------------  -----
 0   order_id                       99441 non-null  str  
 1   customer_id                    99441 non-null  str  
 2   order_status                   99441 non-null  str  
 3   order_purchase_timestamp       99441 non-null  str  
 4   order_approved_at              99281 non-null  str  
 5   order_delivered_carrier_date   97658 non-null  str  
 6   order_delivered_customer_date  96476 non-null  str  
 7   order_estimated_delivery_date  99441 non-null  str  
dtypes: str(8)
memory usage: 6.1 MB


In [6]:
date_columns = [
    "order_purchase_timestamp",
    "order_approved_at",
    "order_delivered_carrier_date",
    "order_delivered_customer_date",
    "order_estimated_delivery_date"
]

for column in date_columns:
    orders[column] = pd.to_datetime(
        orders[column],
        errors="coerce"
    )

In [7]:
orders.info()

<class 'pandas.DataFrame'>
RangeIndex: 99441 entries, 0 to 99440
Data columns (total 8 columns):
 #   Column                         Non-Null Count  Dtype         
---  ------                         --------------  -----         
 0   order_id                       99441 non-null  str           
 1   customer_id                    99441 non-null  str           
 2   order_status                   99441 non-null  str           
 3   order_purchase_timestamp       99441 non-null  datetime64[us]
 4   order_approved_at              99281 non-null  datetime64[us]
 5   order_delivered_carrier_date   97658 non-null  datetime64[us]
 6   order_delivered_customer_date  96476 non-null  datetime64[us]
 7   order_estimated_delivery_date  99441 non-null  datetime64[us]
dtypes: datetime64[us](5), str(3)
memory usage: 6.1 MB


# Delivery_days
We created this feature to understand how long does it take Olist to deliver an order measured in days. This step helps us undertand the delays.

In [8]:
orders["delivery_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_purchase_timestamp"]
).dt.days

In [9]:
orders["delivery_days"].describe()

count    96476.000000
mean        12.094086
std          9.551746
min          0.000000
25%          6.000000
50%         10.000000
75%         15.000000
max        209.000000
Name: delivery_days, dtype: float64

In [10]:
orders["delivery_delay_days"] = (
    orders["order_delivered_customer_date"]
    - orders["order_estimated_delivery_date"]
).dt.days

In [11]:
orders["is_late_delivery"] = (
    orders["delivery_delay_days"] > 0
)

# Relationship Validation

## Objective

Validate that primary and foreign key relationships are consistent across the datasets before building SQL models and dashboards.

In [12]:
missing_customers = (
    ~orders["customer_id"].isin(customers["customer_id"])
).sum()

print(f"Orders without matching customer: {missing_customers}")

Orders without matching customer: 0


In [13]:
missing_orders = (
    ~items["order_id"].isin(orders["order_id"])
).sum()

print(f"Order items without matching order: {missing_orders}")

Order items without matching order: 0


In [14]:
missing_payments = (
    ~payments["order_id"].isin(orders["order_id"])
).sum()

print(f"Payments without matching order: {missing_payments}")

Payments without matching order: 0


In [15]:
missing_reviews = (
    ~reviews["order_id"].isin(orders["order_id"])
).sum()

print(f"Reviews without matching order: {missing_reviews}")

Reviews without matching order: 0


In [16]:
missing_sellers = (
    ~items["seller_id"].isin(sellers["seller_id"])
).sum()

print(f"Unknown sellers: {missing_sellers}")

Unknown sellers: 0


In [17]:
missing_products = (
    ~items["product_id"].isin(products["product_id"])
).sum()

print(f"Unknown products: {missing_products}")

Unknown products: 0


In [18]:
validation_df = pd.DataFrame({
    "Relationship": [
        "Orders → Customers",
        "Items → Orders",
        "Payments → Orders",
        "Reviews → Orders",
        "Items → Sellers",
        "Items → Products"
    ],
    "Missing Records": [
        missing_customers,
        missing_orders,
        missing_payments,
        missing_reviews,
        missing_sellers,
        missing_products
    ]
})

validation_df

,Relationship,Missing Records
0,Orders → Customers,0
1,Items → Orders,0
2,Payments → Orders,0
3,Reviews → Orders,0
4,Items → Sellers,0
5,Items → Products,0


# Relationship Validation Summary

## Findings

All major relationships between the transactional and dimension datasets were successfully validated. No orphaned records were identified, indicating there is a strong referential integrity across the source data.

This provides confidence that the datasets can be joined reliably for SQL modeling, business analysis, and dashboard development.

# Feature Engineering

## Objective

Create additional variables that make business analysis easier and improve downstream SQL queries, dashboards, and machine learning models.

In [19]:
orders["order_year"] = (
    orders["order_purchase_timestamp"]
    .dt.year
)

In [20]:
orders["order_month"] = (
    orders["order_purchase_timestamp"]
    .dt.month
)

In [21]:
orders["month_name"] = (
    orders["order_purchase_timestamp"]
    .dt.month_name()
)

In [22]:
orders["order_quarter"] = (
    orders["order_purchase_timestamp"]
    .dt.quarter
)

In [23]:
orders["weekday"] = (
    orders["order_purchase_timestamp"]
    .dt.day_name()
)

In [24]:
orders["is_weekend_purchase"] = (
    orders["order_purchase_timestamp"]
    .dt.weekday >= 5
)

In [25]:
orders["purchase_hour"] = (
    orders["order_purchase_timestamp"]
    .dt.hour
)

In [26]:
orders[
    [
        "order_purchase_timestamp",
        "order_year",
        "order_month",
        "month_name",
        "order_quarter",
        "weekday",
        "purchase_hour",
        "is_weekend_purchase"
    ]
].head(4)

,order_purchase_timestamp,order_year,order_month,month_name,order_quarter,weekday,purchase_hour,is_weekend_purchase
0,2017-10-02 10:56:33,2017,10,October,4,Monday,10,False
1,2018-07-24 20:41:37,2018,7,July,3,Tuesday,20,False
2,2018-08-08 08:38:49,2018,8,August,3,Wednesday,8,False
3,2017-11-18 19:28:06,2017,11,November,4,Saturday,19,True


# Feature Engineering Summary

## Business Value

The newly engineered features simplify time-based analysis and reduce repetitive calculations in downstream SQL queries and Power BI dashboards.

These variables enable analyses such as:

- Monthly revenue trends
- Quarterly performance
- Peak purchasing hours
- Weekday versus weekend purchasing behavior
- Seasonal demand patterns